# Multimodal Document Intelligence RAG (OpenAI)

End-to-end pipeline that indexes PDFs by **both** their text and their visual content
(charts, tables, diagrams, scanned pages), then answers questions with page-level citations
and a grounded-answer evaluation loop.

**Why this design.** Text-only RAG silently drops everything that lives in figures. Pure CLIP-style
joint embedding handles images but is weak on dense document semantics. This pipeline uses
*vision-to-text transcription at index time*: a VLM converts each page's visual content into a
dense textual surrogate, which is embedded in the same space as the extracted text. Retrieval
stays in one unified text space (cheap, hybrid-friendly, debuggable); the *generation* step is
multimodal, re-attaching the original page bitmaps for the top-ranked pages so the model can read
the actual axis labels rather than trusting the surrogate.

**Model choices (as of Aug 2026).** The GPT-5.6 family is natively multimodal and served through
the Responses API. `gpt-5.6-luna` handles the high-volume per-page extraction, `gpt-5.6-sol` (or
`-terra`) handles synthesis and judging. Embeddings remain `text-embedding-3-small/large`.
Docs: https://developers.openai.com/api/docs/models

---

## 0. Environment

`faiss` and `tiktoken` are optional — the notebook degrades gracefully without them.

In [1]:
# !pip install -q "openai>=1.60" pymupdf pydantic numpy rank-bm25 pillow tiktoken faiss-cpu

from __future__ import annotations

import base64
import concurrent.futures as cf
import io
import json
import logging
import math
import os
import random
import re
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable, Iterable, Literal, Protocol, Sequence, TypeVar

import numpy as np
from pydantic import BaseModel, Field, ValidationError

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
)
logger = logging.getLogger("mm_rag")
logging.getLogger("httpx").setLevel(logging.WARNING)

## 1. Configuration

Everything tunable lives in one frozen dataclass. Do not scatter model names through the codebase —
model deprecation is the single most common cause of a dead notebook six months later.

In [2]:
@dataclass(frozen=True)
class Config:
    # --- models -------------------------------------------------------------
    vision_model: str = "gpt-5.6-luna"       # per-page extraction: high volume, cost-sensitive
    synthesis_model: str = "gpt-5.6-terra"   # answer generation: needs reasoning over many pages
    judge_model: str = "gpt-5.6-sol"         # eval judge: use the strongest model you can afford
    embedding_model: str = "text-embedding-3-small"
    embedding_dims: int = 1024               # Matryoshka truncation; 1536 is the native size

    # --- reasoning effort (GPT-5.6 family) -----------------------------------
    vision_effort: Literal["none", "low", "medium", "high"] = "low"
    synthesis_effort: Literal["none", "low", "medium", "high"] = "medium"

    # --- ingestion ------------------------------------------------------------
    render_dpi: int = 150                    # 150 is the accuracy/token sweet spot for A4
    max_image_edge_px: int = 1536            # downscale before base64 to cap vision tokens
    text_poor_char_threshold: int = 180      # below this, treat the page as scanned/graphical

    # --- chunking -------------------------------------------------------------
    target_chunk_tokens: int = 380
    chunk_overlap_tokens: int = 60
    semantic_breakpoint_percentile: float = 88.0  # cosine-distance percentile for a split

    # --- retrieval ------------------------------------------------------------
    dense_top_k: int = 12
    sparse_top_k: int = 12
    rrf_k: int = 60                          # Cormack et al. RRF constant
    final_top_k: int = 6
    max_pages_in_context: int = 3            # page bitmaps re-attached at synthesis time

    # --- runtime --------------------------------------------------------------
    max_workers: int = 6
    max_retries: int = 5
    request_timeout_s: float = 120.0

    # --- cost accounting (USD per 1M tokens) ----------------------------------
    # Verify against https://developers.openai.com/api/docs/pricing before trusting the totals.
    prices: dict[str, tuple[float, float]] = field(
        default_factory=lambda: {
            "gpt-5.6-sol": (5.00, 30.00),
            "gpt-5.6-terra": (1.25, 10.00),
            "gpt-5.6-luna": (0.25, 2.00),
            "text-embedding-3-small": (0.02, 0.0),
            "text-embedding-3-large": (0.13, 0.0),
        }
    )


CFG = Config()

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError(
        "OPENAI_API_KEY is not set. Export it in your shell or load it from a .env file; "
        "never hardcode it in a notebook you intend to commit."
    )

## 2. Instrumented client

Every call goes through one wrapper that owns retries, timeouts, and usage accounting.
Without this you cannot answer "what did that ingest run cost?" — which is the first question
anyone asks when this moves from notebook to service.

In [3]:
from openai import (
    APIConnectionError,
    APITimeoutError,
    InternalServerError,
    OpenAI,
    RateLimitError,
)

T = TypeVar("T")

RETRYABLE = (RateLimitError, APITimeoutError, APIConnectionError, InternalServerError)


@dataclass
class UsageLedger:
    """Thread-safe-enough accumulator (GIL-protected += on floats/ints is fine here)."""

    rows: list[dict[str, Any]] = field(default_factory=list)

    def record(self, model: str, in_tok: int, out_tok: int, latency_s: float) -> None:
        p_in, p_out = CFG.prices.get(model, (0.0, 0.0))
        self.rows.append(
            {
                "model": model,
                "input_tokens": in_tok,
                "output_tokens": out_tok,
                "latency_s": round(latency_s, 3),
                "cost_usd": (in_tok / 1e6) * p_in + (out_tok / 1e6) * p_out,
            }
        )

    def report(self) -> dict[str, Any]:
        if not self.rows:
            return {"calls": 0, "cost_usd": 0.0}
        lat = sorted(r["latency_s"] for r in self.rows)
        by_model: dict[str, dict[str, float]] = {}
        for r in self.rows:
            m = by_model.setdefault(r["model"], {"calls": 0, "in": 0, "out": 0, "usd": 0.0})
            m["calls"] += 1
            m["in"] += r["input_tokens"]
            m["out"] += r["output_tokens"]
            m["usd"] += r["cost_usd"]
        return {
            "calls": len(self.rows),
            "cost_usd": round(sum(r["cost_usd"] for r in self.rows), 4),
            "latency_p50_s": lat[len(lat) // 2],
            "latency_p95_s": lat[min(len(lat) - 1, int(0.95 * len(lat)))],
            "by_model": {k: {**v, "usd": round(v["usd"], 4)} for k, v in by_model.items()},
        }


LEDGER = UsageLedger()
client = OpenAI(timeout=CFG.request_timeout_s, max_retries=0)  # retries handled explicitly below


def with_retry(fn: Callable[[], T], *, what: str) -> T:
    """Exponential backoff with full jitter. Only retries transport/throttle errors —
    a 400 from a malformed schema must fail loudly and immediately."""
    last: Exception | None = None
    for attempt in range(CFG.max_retries):
        try:
            return fn()
        except RETRYABLE as exc:
            last = exc
            sleep_s = random.uniform(0, min(30.0, 2.0 ** attempt))
            logger.warning(
                "%s failed (attempt %d/%d): %s — retrying in %.1fs",
                what, attempt + 1, CFG.max_retries, type(exc).__name__, sleep_s,
            )
            time.sleep(sleep_s)
    raise RuntimeError(f"{what} failed after {CFG.max_retries} attempts") from last


def parse_structured(
    *,
    model: str,
    schema: type[BaseModel],
    instructions: str,
    content: list[dict[str, Any]],
    effort: str | None = None,
) -> BaseModel:
    """Responses API call with a strict JSON schema derived from a Pydantic model."""
    kwargs: dict[str, Any] = {
        "model": model,
        "instructions": instructions,
        "input": [{"role": "user", "content": content}],
        "text_format": schema,
    }
    if effort:
        kwargs["reasoning"] = {"effort": effort}

    t0 = time.perf_counter()
    resp = with_retry(lambda: client.responses.parse(**kwargs), what=f"{model}:{schema.__name__}")
    LEDGER.record(model, resp.usage.input_tokens, resp.usage.output_tokens, time.perf_counter() - t0)

    parsed = resp.output_parsed
    if parsed is None:  # refusal or truncation
        raise ValueError(f"{model} returned no parsed output (status={resp.status!r})")
    return parsed


def embed_texts(texts: Sequence[str], *, batch_size: int = 96) -> np.ndarray:
    """Returns L2-normalised float32 embeddings, shape (len(texts), embedding_dims)."""
    if not texts:
        return np.zeros((0, CFG.embedding_dims), dtype=np.float32)

    vectors: list[list[float]] = []
    for i in range(0, len(texts), batch_size):
        batch = [t.replace("\n", " ").strip() or " " for t in texts[i : i + batch_size]]
        t0 = time.perf_counter()
        resp = with_retry(
            lambda: client.embeddings.create(
                model=CFG.embedding_model, input=batch, dimensions=CFG.embedding_dims
            ),
            what="embeddings",
        )
        LEDGER.record(CFG.embedding_model, resp.usage.prompt_tokens, 0, time.perf_counter() - t0)
        vectors.extend(d.embedding for d in sorted(resp.data, key=lambda d: d.index))

    arr = np.asarray(vectors, dtype=np.float32)
    norms = np.linalg.norm(arr, axis=1, keepdims=True)
    return arr / np.clip(norms, 1e-9, None)

## 3. Ingestion — text layer + page rasters

Two things are extracted per page: the embedded text layer, and a rendered bitmap. Pages whose
text layer is thin are flagged `text_poor` — these are scans, slides, or full-page figures where
the VLM is the *only* source of signal, and where a text-only pipeline would return nothing.

Downscaling before base64 matters: vision token cost scales with pixel area, and a 300-DPI A4 page
is ~8.7 MP of mostly whitespace.

In [5]:
import fitz  # PyMuPDF
from PIL import Image


class PageAsset(BaseModel):
    doc_id: str
    page_no: int  # 1-indexed
    text: str
    image_b64: str
    width: int
    height: int
    text_poor: bool


def _downscale_png(png_bytes: bytes, max_edge: int) -> tuple[bytes, int, int]:
    with Image.open(io.BytesIO(png_bytes)) as im:
        im = im.convert("RGB")
        if max(im.size) > max_edge:
            scale = max_edge / max(im.size)
            im = im.resize((int(im.width * scale), int(im.height * scale)), Image.LANCZOS)
        buf = io.BytesIO()
        im.save(buf, format="PNG", optimize=True)
        return buf.getvalue(), im.width, im.height


def load_pdf(path: str | Path, *, doc_id: str | None = None) -> list[PageAsset]:
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"PDF not found: {path}")
    doc_id = doc_id or path.stem

    assets: list[PageAsset] = []
    with fitz.open(path) as doc:
        zoom = CFG.render_dpi / 72.0
        matrix = fitz.Matrix(zoom, zoom)
        for idx, page in enumerate(doc, start=1):
            try:
                text = page.get_text("text") or ""
                png, w, h = _downscale_png(
                    page.get_pixmap(matrix=matrix, alpha=False).tobytes("png"),
                    CFG.max_image_edge_px,
                )
            except Exception:
                logger.exception("Failed to render %s page %d — skipping", doc_id, idx)
                continue

            assets.append(
                PageAsset(
                    doc_id=doc_id,
                    page_no=idx,
                    text=text.strip(),
                    image_b64=base64.b64encode(png).decode("ascii"),
                    width=w,
                    height=h,
                    text_poor=len(text.strip()) < CFG.text_poor_char_threshold,
                )
            )

    logger.info(
        "Loaded %s: %d pages (%d text-poor)",
        doc_id, len(assets), sum(a.text_poor for a in assets),
    )
    return assets


def image_part(b64: str, *, detail: Literal["low", "high", "auto"] = "high") -> dict[str, Any]:
    """Responses API image content part. `high` detail is required for dense document pages —
    `low` downsamples to a thumbnail and will lose axis labels and table rules."""
    return {"type": "input_image", "image_url": f"data:image/png;base64,{b64}", "detail": detail}


def text_part(text: str) -> dict[str, Any]:
    return {"type": "input_text", "text": text}

## 4. Vision-based page understanding

The extraction schema is deliberately narrow. Open-ended "describe this page" prompts produce
fluent, unfaithful prose. Constraining the model to enumerate figures and transcribe tables into
markdown gives output that is (a) verifiable against the page and (b) directly chunkable.

`visual_only` is the important field: it marks content that exists *nowhere* in the text layer,
which is exactly the recall a text-only baseline is missing.

In [6]:
class Figure(BaseModel):
    kind: Literal["chart", "diagram", "photo", "screenshot", "map", "equation", "other"]
    caption: str = Field(description="Caption as printed, or empty string if none.")
    description: str = Field(
        description="What the figure shows, including concrete axis labels, units, series names, "
        "and the direction/magnitude of any trend. Do not speculate beyond what is drawn."
    )
    stated_values: list[str] = Field(
        default_factory=list,
        description="Numeric values legible in the figure, e.g. 'Q3 2025 revenue: 4.2bn'.",
    )


class Table(BaseModel):
    caption: str = ""
    markdown: str = Field(description="Full table transcribed as GitHub-flavoured markdown.")


class PageExtraction(BaseModel):
    summary: str = Field(description="2-4 sentences on what this page covers.")
    section_heading: str = Field(default="", description="Nearest heading, or empty string.")
    figures: list[Figure] = Field(default_factory=list)
    tables: list[Table] = Field(default_factory=list)
    entities: list[str] = Field(
        default_factory=list, description="Named entities, tickers, product names, statute refs."
    )
    visual_only: bool = Field(
        description="True if meaningful content appears only in the imagery, not the text layer."
    )


VISION_INSTRUCTIONS = """You extract structured content from a single page of a document.

Rules:
- Transcribe, do not interpret. Every number you emit must be legible on the page.
- Transcribe tables completely, preserving row and column order. Use an empty cell for blanks.
- For charts, always name the axes and units, and state the trend in concrete terms.
- If the page is blank, boilerplate, or a divider, return an empty summary and no figures/tables.
- Never invent a caption that is not printed."""


def extract_page(asset: PageAsset) -> tuple[PageAsset, PageExtraction | None]:
    """Vision extraction for one page. Returns None on unrecoverable failure so that a single
    bad page cannot abort a 400-page ingest."""
    hint = (
        "The embedded text layer is sparse or absent — this page is likely a scan or a full-page "
        "graphic. Transcribe everything you can read."
        if asset.text_poor
        else f"Embedded text layer (for cross-checking; do not simply copy it):\n{asset.text[:4000]}"
    )
    content = [
        text_part(f"Document: {asset.doc_id} — page {asset.page_no}\n\n{hint}"),
        image_part(asset.image_b64, detail="high"),
    ]
    try:
        result = parse_structured(
            model=CFG.vision_model,
            schema=PageExtraction,
            instructions=VISION_INSTRUCTIONS,
            content=content,
            effort=CFG.vision_effort,
        )
        return asset, result  # type: ignore[return-value]
    except (ValueError, ValidationError, RuntimeError):
        logger.exception("Vision extraction failed for %s p%d", asset.doc_id, asset.page_no)
        return asset, None


def extract_all(assets: Sequence[PageAsset]) -> list[tuple[PageAsset, PageExtraction]]:
    """Bounded-concurrency fan-out. max_workers should stay well under your TPM headroom —
    high-detail page images are token-heavy and will trip rate limits before they trip RPM."""
    out: list[tuple[PageAsset, PageExtraction]] = []
    with cf.ThreadPoolExecutor(max_workers=CFG.max_workers) as pool:
        for asset, extraction in pool.map(extract_page, assets):
            if extraction is not None:
                out.append((asset, extraction))
    out.sort(key=lambda pair: (pair[0].doc_id, pair[0].page_no))
    logger.info("Extracted %d/%d pages", len(out), len(assets))
    return out

## 5. Chunking

Two chunk families land in the same index:

| Modality | Source | Why it exists |
|---|---|---|
| `text` | PDF text layer, semantically split | Precise wording, quotes, clause-level recall |
| `visual` | VLM figure/table/summary output | The only representation of chart and scan content |

Semantic splitting uses adjacent-sentence embedding distance with a percentile breakpoint
(Kamradt's method), then enforces a token budget so no chunk blows the reranker's context.
Tables are never split — a half-table is worse than no table.

In [7]:
try:
    import tiktoken

    try:
        _ENC = tiktoken.encoding_for_model(CFG.synthesis_model)
    except KeyError:
        _ENC = tiktoken.get_encoding("o200k_base")

    def n_tokens(text: str) -> int:
        return len(_ENC.encode(text, disallowed_special=()))
except ImportError:  # pragma: no cover
    logger.warning("tiktoken unavailable — falling back to a 4-chars-per-token heuristic")

    def n_tokens(text: str) -> int:
        return max(1, len(text) // 4)


_SENT_RE = re.compile(r"(?<=[.!?])\s+(?=[A-Z(\"'\[])|\n{2,}")


class Chunk(BaseModel):
    chunk_id: str
    doc_id: str
    page_no: int
    modality: Literal["text", "visual"]
    kind: str  # prose | table | figure | summary
    content: str
    section: str = ""

    def for_embedding(self) -> str:
        """Prefixing with provenance measurably improves retrieval on multi-document corpora:
        it lets the query embedding key on section and document identity, not just body text."""
        head = f"[{self.doc_id} p.{self.page_no}"
        if self.section:
            head += f" — {self.section}"
        return f"{head}] {self.content}"


def _split_sentences(text: str) -> list[str]:
    return [s.strip() for s in _SENT_RE.split(text) if s and s.strip()]


def semantic_split(text: str) -> list[str]:
    """Embedding-distance breakpoint splitting with a hard token cap."""
    sentences = _split_sentences(text)
    if len(sentences) < 4:
        return [text] if text.strip() else []

    vecs = embed_texts(sentences)
    # cosine distance between adjacent sentences (vectors are already normalised)
    distances = 1.0 - np.sum(vecs[:-1] * vecs[1:], axis=1)
    threshold = float(np.percentile(distances, CFG.semantic_breakpoint_percentile))

    chunks: list[str] = []
    buf: list[str] = []
    budget = 0
    for i, sent in enumerate(sentences):
        tok = n_tokens(sent)
        breakpoint_hit = i > 0 and distances[i - 1] > threshold
        if buf and (budget + tok > CFG.target_chunk_tokens or breakpoint_hit):
            chunks.append(" ".join(buf))
            # token-overlap carry-over preserves cross-boundary context
            carry, carried = [], 0
            for prev in reversed(buf):
                if carried >= CFG.chunk_overlap_tokens:
                    break
                carry.insert(0, prev)
                carried += n_tokens(prev)
            buf, budget = carry, carried
        buf.append(sent)
        budget += tok

    if buf:
        chunks.append(" ".join(buf))
    return chunks


def build_chunks(pages: Sequence[tuple[PageAsset, PageExtraction]]) -> list[Chunk]:
    chunks: list[Chunk] = []

    def add(asset: PageAsset, modality: str, kind: str, content: str, section: str) -> None:
        content = content.strip()
        if len(content) < 25:
            return
        chunks.append(
            Chunk(
                chunk_id=f"{asset.doc_id}:{asset.page_no}:{kind}:{len(chunks)}",
                doc_id=asset.doc_id,
                page_no=asset.page_no,
                modality=modality,  # type: ignore[arg-type]
                kind=kind,
                content=content,
                section=section,
            )
        )

    for asset, ext in pages:
        section = ext.section_heading

        for piece in semantic_split(asset.text):
            add(asset, "text", "prose", piece, section)

        add(asset, "visual", "summary", ext.summary, section)

        for fig in ext.figures:
            body = f"{fig.kind.upper()}"
            if fig.caption:
                body += f" — {fig.caption}"
            body += f"\n{fig.description}"
            if fig.stated_values:
                body += "\nValues: " + "; ".join(fig.stated_values)
            add(asset, "visual", "figure", body, section)

        for tbl in ext.tables:
            add(asset, "visual", "table", f"{tbl.caption}\n{tbl.markdown}".strip(), section)

    logger.info(
        "Built %d chunks (%d text, %d visual)",
        len(chunks),
        sum(c.modality == "text" for c in chunks),
        sum(c.modality == "visual" for c in chunks),
    )
    return chunks

## 6. Index — dense + sparse behind one interface

The `VectorIndex` protocol is the seam you will replace in production (Qdrant, Azure AI Search,
pgvector). Brute-force NumPy is exact and fine to a few hundred thousand vectors; a FAISS
implementation of the identical interface is included so the swap is a one-line change.

In [9]:
class VectorIndex(Protocol):
    def add(self, vectors: np.ndarray, ids: Sequence[str]) -> None: ...
    def search(self, query: np.ndarray, k: int) -> list[tuple[str, float]]: ...


class NumpyIndex:
    """Exact inner-product search. Vectors must be L2-normalised, so IP == cosine."""

    def __init__(self, dim: int) -> None:
        self.dim = dim
        self._vectors = np.zeros((0, dim), dtype=np.float32)
        self._ids: list[str] = []

    def add(self, vectors: np.ndarray, ids: Sequence[str]) -> None:
        if vectors.shape[0] != len(ids):
            raise ValueError(f"vector/id mismatch: {vectors.shape[0]} vs {len(ids)}")
        if vectors.shape[1] != self.dim:
            raise ValueError(f"expected dim {self.dim}, got {vectors.shape[1]}")
        self._vectors = np.vstack([self._vectors, vectors.astype(np.float32)])
        self._ids.extend(ids)

    def search(self, query: np.ndarray, k: int) -> list[tuple[str, float]]:
        if not self._ids:
            return []
        scores = self._vectors @ query.astype(np.float32).ravel()
        top = np.argsort(-scores)[: min(k, len(self._ids))]
        return [(self._ids[i], float(scores[i])) for i in top]


class FaissIndex:
    """Drop-in replacement once the corpus outgrows brute force."""

    def __init__(self, dim: int) -> None:
        import faiss  # noqa: PLC0415

        self._index = faiss.IndexFlatIP(dim)
        self._ids: list[str] = []

    def add(self, vectors: np.ndarray, ids: Sequence[str]) -> None:
        self._index.add(np.ascontiguousarray(vectors, dtype=np.float32))
        self._ids.extend(ids)

    def search(self, query: np.ndarray, k: int) -> list[tuple[str, float]]:
        scores, idx = self._index.search(
            np.ascontiguousarray(query.reshape(1, -1), dtype=np.float32), min(k, len(self._ids))
        )
        return [(self._ids[i], float(s)) for i, s in zip(idx[0], scores[0]) if i != -1]


from rank_bm25 import BM25Okapi

_TOKEN_RE = re.compile(r"[a-z0-9]+")


def tokenize(text: str) -> list[str]:
    return _TOKEN_RE.findall(text.lower())


@dataclass
class Corpus:
    chunks: list[Chunk]
    index: VectorIndex
    bm25: BM25Okapi
    by_id: dict[str, Chunk]
    pages: dict[tuple[str, int], PageAsset]


def build_corpus(
    chunks: Sequence[Chunk], page_assets: Sequence[PageAsset], *, use_faiss: bool = False
) -> Corpus:
    if not chunks:
        raise ValueError("Cannot build a corpus from zero chunks")

    vectors = embed_texts([c.for_embedding() for c in chunks])
    index: VectorIndex = (
        FaissIndex(CFG.embedding_dims) if use_faiss else NumpyIndex(CFG.embedding_dims)
    )
    index.add(vectors, [c.chunk_id for c in chunks])

    return Corpus(
        chunks=list(chunks),
        index=index,
        bm25=BM25Okapi([tokenize(c.for_embedding()) for c in chunks]),
        by_id={c.chunk_id: c for c in chunks},
        pages={(p.doc_id, p.page_no): p for p in page_assets},
    )

## 7. Hybrid retrieval with Reciprocal Rank Fusion

RRF fuses the dense and sparse rankings without needing score normalisation between two
incomparable scales — the usual failure mode of naive weighted-sum hybrid search. Exact identifiers
(statute numbers, SKUs, tickers) are where BM25 earns its place; dense retrieval reliably misses them.

In [10]:
class Retrieved(BaseModel):
    chunk: Chunk
    score: float
    dense_rank: int | None = None
    sparse_rank: int | None = None


def hybrid_search(corpus: Corpus, query: str, *, top_k: int | None = None) -> list[Retrieved]:
    if not query.strip():
        raise ValueError("Empty query")
    top_k = top_k or CFG.final_top_k

    qvec = embed_texts([query])[0]
    dense = corpus.index.search(qvec, CFG.dense_top_k)

    bm25_scores = corpus.bm25.get_scores(tokenize(query))
    sparse_idx = np.argsort(-bm25_scores)[: CFG.sparse_top_k]
    sparse = [(corpus.chunks[i].chunk_id, float(bm25_scores[i])) for i in sparse_idx]

    dense_rank = {cid: r for r, (cid, _) in enumerate(dense, start=1)}
    sparse_rank = {cid: r for r, (cid, _) in enumerate(sparse, start=1)}

    fused: dict[str, float] = {}
    for ranks in (dense_rank, sparse_rank):
        for cid, rank in ranks.items():
            fused[cid] = fused.get(cid, 0.0) + 1.0 / (CFG.rrf_k + rank)

    ordered = sorted(fused.items(), key=lambda kv: -kv[1])[:top_k]
    return [
        Retrieved(
            chunk=corpus.by_id[cid],
            score=score,
            dense_rank=dense_rank.get(cid),
            sparse_rank=sparse_rank.get(cid),
        )
        for cid, score in ordered
    ]

## 8. Multimodal answer synthesis

The retrieved *text* is what was searched; the retrieved *pages* are what gets read. For the
highest-ranked pages the original bitmap is re-attached, so the model verifies the VLM's
index-time transcription against the actual page before answering. This is the step that catches
transcription drift — the main risk of a describe-then-index architecture.

Citations are a structured field, not a formatting convention. A model asked to "cite your sources"
in prose will happily fabricate a plausible page number; a schema-constrained integer that you
validate against the retrieved set will not survive the check.

In [11]:
class Citation(BaseModel):
    doc_id: str
    page_no: int
    quote: str = Field(description="Short verbatim span from the source supporting the claim.")


class GroundedAnswer(BaseModel):
    answer: str = Field(description="Direct answer. State plainly if the context is insufficient.")
    citations: list[Citation] = Field(default_factory=list)
    sufficient_context: bool = Field(
        description="False if the provided context does not contain the answer."
    )
    used_visual_evidence: bool = Field(
        description="True if a chart, table image, or scan was needed to answer."
    )


SYNTHESIS_INSTRUCTIONS = """You answer questions strictly from the supplied document context.

Rules:
- Use only the provided text excerpts and page images. Never use outside knowledge.
- If the context is insufficient, set sufficient_context to false and say so in the answer.
  A wrong confident answer is far more costly than an admitted gap.
- Cite every factual claim with the document id and page number it came from.
- When an excerpt and a page image disagree, trust the image and answer from it.
- Preserve exact figures, units, and dates as printed."""


def answer_question(corpus: Corpus, question: str) -> tuple[GroundedAnswer, list[Retrieved]]:
    hits = hybrid_search(corpus, question)
    if not hits:
        return (
            GroundedAnswer(
                answer="No indexed content matched this question.",
                sufficient_context=False,
                used_visual_evidence=False,
            ),
            [],
        )

    excerpts = "\n\n".join(
        f"[{i}] {h.chunk.doc_id} p.{h.chunk.page_no} ({h.chunk.modality}/{h.chunk.kind})\n"
        f"{h.chunk.content}"
        for i, h in enumerate(hits, start=1)
    )

    # de-duplicate pages, preserving fusion order
    page_keys: list[tuple[str, int]] = []
    for h in hits:
        key = (h.chunk.doc_id, h.chunk.page_no)
        if key not in page_keys:
            page_keys.append(key)
    page_keys = page_keys[: CFG.max_pages_in_context]

    content: list[dict[str, Any]] = [
        text_part(f"Question: {question}\n\nRetrieved excerpts:\n{excerpts}")
    ]
    for doc_id, page_no in page_keys:
        page = corpus.pages.get((doc_id, page_no))
        if page is None:
            continue
        content.append(text_part(f"Full page image — {doc_id} p.{page_no}:"))
        content.append(image_part(page.image_b64, detail="high"))

    result = parse_structured(
        model=CFG.synthesis_model,
        schema=GroundedAnswer,
        instructions=SYNTHESIS_INSTRUCTIONS,
        content=content,
        effort=CFG.synthesis_effort,
    )
    assert isinstance(result, GroundedAnswer)

    # Guardrail: a citation to a page that was never retrieved is a hallucination, not an answer.
    retrieved_pages = {(h.chunk.doc_id, h.chunk.page_no) for h in hits}
    for cite in result.citations:
        if (cite.doc_id, cite.page_no) not in retrieved_pages:
            logger.error(
                "Fabricated citation: %s p.%d not in retrieved set", cite.doc_id, cite.page_no
            )
    return result, hits

## 9. Evaluation

Two layers, because they fail independently:

1. **Retrieval** — recall@k and MRR against gold page labels. If recall@k is low, no amount of
   prompt work on the generator will help.
2. **Generation** — an LLM judge scoring groundedness and answer correctness, plus a deterministic
   citation-precision check that needs no model at all.

Judge scores are only meaningful once you have spot-checked the judge against your own labels on
~30 examples. Treat them as a regression signal, never as ground truth.

In [12]:
class EvalCase(BaseModel):
    question: str
    gold_pages: list[int]
    gold_answer: str = ""
    requires_visual: bool = False


class JudgeVerdict(BaseModel):
    groundedness: int = Field(ge=1, le=5, description="5 = every claim traceable to the context.")
    correctness: int = Field(ge=1, le=5, description="5 = matches the reference answer.")
    rationale: str


JUDGE_INSTRUCTIONS = """You grade a RAG answer. Be strict and terse.

groundedness: does every claim follow from the retrieved context? Unsupported claims cap this at 2.
correctness: does the answer match the reference? Missing key figures cap this at 3.
Judge only what is written. Do not reward fluent prose."""


def judge_answer(case: EvalCase, answer: GroundedAnswer, hits: Sequence[Retrieved]) -> JudgeVerdict:
    context = "\n\n".join(f"{h.chunk.doc_id} p.{h.chunk.page_no}: {h.chunk.content}" for h in hits)
    payload = (
        f"Question:\n{case.question}\n\n"
        f"Reference answer:\n{case.gold_answer or '(none supplied)'}\n\n"
        f"Retrieved context:\n{context}\n\n"
        f"Answer under test:\n{answer.answer}"
    )
    result = parse_structured(
        model=CFG.judge_model,
        schema=JudgeVerdict,
        instructions=JUDGE_INSTRUCTIONS,
        content=[text_part(payload)],
        effort="low",
    )
    assert isinstance(result, JudgeVerdict)
    return result


def evaluate(corpus: Corpus, cases: Sequence[EvalCase], *, k: int = 6) -> dict[str, Any]:
    rows: list[dict[str, Any]] = []

    for case in cases:
        try:
            answer, hits = answer_question(corpus, case.question)
        except Exception:
            logger.exception("Eval case failed: %s", case.question[:60])
            continue

        retrieved_pages = [h.chunk.page_no for h in hits][:k]
        gold = set(case.gold_pages)
        hit = bool(gold & set(retrieved_pages))
        rr = next(
            (1.0 / r for r, p in enumerate(retrieved_pages, start=1) if p in gold), 0.0
        )

        cited = {(c.doc_id, c.page_no) for c in answer.citations}
        valid = {(h.chunk.doc_id, h.chunk.page_no) for h in hits}
        citation_precision = len(cited & valid) / len(cited) if cited else 0.0

        verdict = judge_answer(case, answer, hits)
        rows.append(
            {
                "question": case.question,
                "recall_at_k": int(hit),
                "mrr": rr,
                "citation_precision": citation_precision,
                "groundedness": verdict.groundedness,
                "correctness": verdict.correctness,
                "sufficient_context": answer.sufficient_context,
                "visual_expected": case.requires_visual,
                "visual_used": answer.used_visual_evidence,
            }
        )

    if not rows:
        return {"n": 0}

    def mean(key: str) -> float:
        return round(sum(float(r[key]) for r in rows) / len(rows), 3)

    visual_cases = [r for r in rows if r["visual_expected"]]
    return {
        "n": len(rows),
        f"recall@{k}": mean("recall_at_k"),
        "mrr": mean("mrr"),
        "citation_precision": mean("citation_precision"),
        "groundedness_mean": mean("groundedness"),
        "correctness_mean": mean("correctness"),
        "visual_recall": (
            round(sum(r["recall_at_k"] for r in visual_cases) / len(visual_cases), 3)
            if visual_cases
            else None
        ),
        "rows": rows,
    }

## 10. Run it

Point `PDF_PATH` at a document with real charts or tables — an annual report, a scientific paper,
a slide deck export. A pure-prose document will not exercise the multimodal path and the ablation
in section 11 will show no gap.

In [14]:
PDF_PATH = "/Users/manuvenugopalan/Documents/Programming/AgenticAI/northwind_fy2025_sample.pdf" 

pages = load_pdf(PDF_PATH)
extractions = extract_all(pages)
chunks = build_chunks(extractions)
corpus = build_corpus(chunks, pages)

print(json.dumps(LEDGER.report(), indent=2))

2026-08-15 22:30:52,355 | INFO     | mm_rag | Loaded northwind_fy2025_sample: 6 pages (1 text-poor)
2026-08-15 22:30:57,473 | INFO     | mm_rag | Extracted 6/6 pages
2026-08-15 22:31:00,230 | INFO     | mm_rag | Built 22 chunks (12 text, 10 visual)


{
  "calls": 12,
  "cost_usd": 0.007,
  "latency_p50_s": 3.362,
  "latency_p95_s": 5.116,
  "by_model": {
    "gpt-5.6-luna": {
      "calls": 6,
      "in": 15601,
      "out": 1533,
      "usd": 0.007
    },
    "text-embedding-3-small": {
      "calls": 6,
      "in": 2874,
      "out": 0,
      "usd": 0.0001
    }
  }
}


In [15]:
QUESTION = "What does the main chart show, and what are its axis units?"

answer, hits = answer_question(corpus, QUESTION)

print(answer.answer)
print("\nsufficient_context:", answer.sufficient_context, "| visual:", answer.used_visual_evidence)
print("\nCitations:")
for c in answer.citations:
    print(f"  {c.doc_id} p.{c.page_no}: {c.quote[:110]}")
print("\nRetrieval trace:")
for h in hits:
    print(
        f"  {h.score:.4f} | {h.chunk.modality:<6} {h.chunk.kind:<7} | p.{h.chunk.page_no} "
        f"| dense={h.dense_rank} sparse={h.sparse_rank} | {h.chunk.content[:70]!r}"
    )

The main chart (Figure 2) is a bar chart of FY2025 gross margin by product line: Retail Analytics (41.2%), Wholesale Data (28.7%), Digital Subscriptions (63.5%), and Professional Services (22.4%). It also includes a dashed group-average reference line at 38.9%. The x-axis consists of product-line categories, and the y-axis is Gross margin, measured in percent (%).

sufficient_context: True | visual: True

Citations:
  northwind_fy2025_sample p.4: Figure 2: Gross margin by product line, FY2025
  northwind_fy2025_sample p.4: Gross margin (%)
  northwind_fy2025_sample p.4: Group avg 38.9%

Retrieval trace:
  0.0328 | visual figure  | p.4 | dense=1 sparse=1 | 'CHART — Figure 2: Gross margin by product line, FY2025\nBar chart with '
  0.0320 | visual summary | p.4 | dense=2 sparse=3 | 'This page presents gross margin analysis by product line for FY2025. I'
  0.0313 | visual figure  | p.2 | dense=6 sparse=2 | 'CHART — Figure 1: Monthly active users, FY2025\nLine chart with x-axis '
  0.0301 

## 11. Ablation — does the visual channel actually earn its cost?

Vision extraction dominates the ingest bill, so justify it with a number rather than an assumption.
This rebuilds the corpus using text chunks only and re-runs the same eval set. On document sets
with real figures, expect the gap to concentrate almost entirely in the `requires_visual` cases.

In [16]:
EVAL_CASES = [
    EvalCase(
        question="What does the main chart show, and what are its axis units?",
        gold_pages=[1],
        gold_answer="",
        requires_visual=True,
    ),
    # Add 30-50 cases before trusting any of these numbers. Under ~30 the confidence
    # interval on recall@k is wider than the effect sizes you are trying to detect.
]

text_only_corpus = build_corpus([c for c in chunks if c.modality == "text"], pages)

full_metrics = evaluate(corpus, EVAL_CASES)
text_metrics = evaluate(text_only_corpus, EVAL_CASES)

print(f"{'metric':<22}{'text-only':>12}{'multimodal':>12}")
for key in ("recall@6", "mrr", "citation_precision", "groundedness_mean", "correctness_mean"):
    print(f"{key:<22}{text_metrics.get(key, 0):>12}{full_metrics.get(key, 0):>12}")

print("\nCost & latency:")
print(json.dumps(LEDGER.report(), indent=2))

metric                   text-only  multimodal
recall@6                       1.0         1.0
mrr                            1.0        0.25
citation_precision             1.0         1.0
groundedness_mean              2.0         5.0
correctness_mean               5.0         5.0

Cost & latency:
{
  "calls": 21,
  "cost_usd": 0.0576,
  "latency_p50_s": 1.946,
  "latency_p95_s": 5.106,
  "by_model": {
    "gpt-5.6-luna": {
      "calls": 6,
      "in": 15601,
      "out": 1533,
      "usd": 0.007
    },
    "text-embedding-3-small": {
      "calls": 10,
      "in": 4147,
      "out": 0,
      "usd": 0.0001
    },
    "gpt-5.6-terra": {
      "calls": 3,
      "in": 21116,
      "out": 575,
      "usd": 0.0321
    },
    "gpt-5.6-sol": {
      "calls": 2,
      "in": 1999,
      "out": 279,
      "usd": 0.0184
    }
  }
}
